# Logistics Transfer Delay Classification Notebook

This notebook starts the implementation of an end-to-end binary classification pipeline to predict whether a transfer is delayed by more than 30 minutes.

**Primary metric:** ROC-AUC

## 1) Data Loading

We load data using relative paths so the notebook can be moved between environments without changing core logic.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

sns.set_theme(style='whitegrid')

data_path = './'
train_file = 'regional_logistics_transfers_train.csv'
test_file = 'regional_logistics_transfers_test.csv'

train_df = pd.read_csv(data_path + train_file)
test_df = pd.read_csv(data_path + test_file)

print(f'Train shape: {train_df.shape}')
print(f'Test shape: {test_df.shape}')

## 2) Exploratory Data Analysis (EDA)

We inspect target balance and missingness to guide preprocessing decisions.

In [ ]:
display(train_df.head())
display(train_df.isna().mean().sort_values(ascending=False).head(10))
print(train_df['is_delayed'].value_counts(normalize=True).rename('class_ratio'))

### Visualization 1: Target distribution

This chart checks whether the delayed class is imbalanced, which affects modeling and threshold interpretation.

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=train_df, x='is_delayed')
plt.title('Target Distribution: On-Time (0) vs Delayed (1)')
plt.xlabel('is_delayed')
plt.ylabel('Count')
plt.show()

**Takeaways / Insights**
- If one class is much smaller, ROC-AUC remains a robust metric for ranking quality.
- Class balance here informs future thresholding decisions, but does not change the ROC-AUC objective.

### Visualization 2: Dispatch delay by target

This chart tests whether operational delay signals separate delayed vs on-time outcomes.

In [ ]:
plt.figure(figsize=(8, 4))
sns.boxplot(data=train_df, x='is_delayed', y='dispatch_delay_minutes')
plt.title('Dispatch Delay Minutes by Target Class')
plt.xlabel('is_delayed')
plt.ylabel('dispatch_delay_minutes')
plt.show()

**Takeaways / Insights**
- A clear upward shift for the delayed class would indicate strong predictive signal in dispatch delays.
- Heavy tails or outliers suggest robust models or regularization may generalize better than overly rigid models.

## 3) Preprocessing

### Assumption: leakage-aware feature filtering
We drop columns that may capture post-event operational outcomes too close to the target definition (for example actual arrival/travel fields) to reduce leakage risk and improve realism for prospective prediction.

### Assumption: missing value treatment
- Numeric features use **median imputation** because it is robust to skew/outliers common in operational delay data.
- Categorical features use **most-frequent imputation** to preserve valid category tokens for one-hot encoding.

In [ ]:
target_col = 'is_delayed'

leakage_columns = [
    'actual_arrival_time',
    'actual_travel_time_minutes',
    'in_transit_time_minutes',
]

train_work = train_df.drop(columns=leakage_columns, errors='ignore').copy()
test_work = test_df.drop(columns=leakage_columns, errors='ignore').copy()

X = train_work.drop(columns=[target_col])
y = train_work[target_col]

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

numeric_cols = X_train.select_dtypes(include=['number']).columns.tolist()
categorical_cols = X_train.select_dtypes(exclude=['number']).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols),
    ]
)

## 4) Feature Engineering

### Assumption: time decomposition helps capture operational periodicity
We transform HHMM-style timestamps into hour/minute parts and extract calendar signals from operation date. This often improves separability while staying interpretable.

In [ ]:
def decompose_hhmm(df, col):
    # We parse HHMM as numeric, preserving NaN, then split into interpretable hour/minute pieces.
    numeric_time = pd.to_numeric(df[col], errors='coerce')
    df[f'{col}_hour'] = (numeric_time // 100).astype('float')
    df[f'{col}_minute'] = (numeric_time % 100).astype('float')

for frame in [X_train, X_valid, test_work]:
    if 'operation_date' in frame.columns:
        op_date = pd.to_datetime(frame['operation_date'], errors='coerce')
        frame['operation_month'] = op_date.dt.month
        frame['operation_day'] = op_date.dt.day

    for time_col in ['scheduled_dispatch_time', 'scheduled_arrival_time', 'actual_dispatch_time']:
        if time_col in frame.columns:
            decompose_hhmm(frame, time_col)

X_train = X_train.drop(columns=['operation_date'], errors='ignore')
X_valid = X_valid.drop(columns=['operation_date'], errors='ignore')
test_model = test_work.drop(columns=['operation_date'], errors='ignore')

numeric_cols = X_train.select_dtypes(include=['number']).columns.tolist()
categorical_cols = X_train.select_dtypes(exclude=['number']).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols),
    ]
)

## 5) Modeling

We begin with an interpretable baseline (Logistic Regression), then keep a failed attempt section as required.

In [ ]:
# We scale only for Logistic Regression to stabilize optimization with mixed feature magnitudes.
log_numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler(with_mean=False)),
])

log_preprocessor = ColumnTransformer(
    transformers=[
        ('num', log_numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols),
    ]
)

logistic_pipeline = Pipeline(steps=[
    ('preprocess', log_preprocessor),
    ('model', LogisticRegression(max_iter=1000, random_state=42))
])

logistic_pipeline.fit(X_train, y_train)
valid_pred_log = logistic_pipeline.predict_proba(X_valid)[:, 1]
log_auc = roc_auc_score(y_valid, valid_pred_log)
print(f'Logistic Regression validation ROC-AUC: {log_auc:.4f}')

### Failed Experiment / Unsuccessful Attempt

We tested a shallow Decision Tree to keep interpretability high, but this model often underperforms on tabular problems with many sparse one-hot categories because single trees can overfit local splits and rank probabilities poorly for ROC-AUC.

In [ ]:
tree_pipeline = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('model', DecisionTreeClassifier(max_depth=4, random_state=42))
])

tree_pipeline.fit(X_train, y_train)
valid_pred_tree = tree_pipeline.predict_proba(X_valid)[:, 1]
tree_auc = roc_auc_score(y_valid, valid_pred_tree)
print(f'Decision Tree validation ROC-AUC: {tree_auc:.4f}')

## 6) Evaluation

We compare validation ROC-AUC and keep the best model for full-train fitting and test prediction.

In [ ]:
model_scores = {
    'logistic_regression': log_auc,
    'decision_tree_failed_attempt': tree_auc,
}

best_name = max(model_scores, key=model_scores.get)
print('Validation ROC-AUC results:', model_scores)
print('Selected model:', best_name)

selected_pipeline = logistic_pipeline if best_name == 'logistic_regression' else tree_pipeline

## 7) Final Training and Submission File

We retrain the selected pipeline on all training rows and create predictions in the exact required format (`transfer_id`, `predict_prob`).

In [ ]:
X_full = train_work.drop(columns=[target_col]).copy()
y_full = train_work[target_col].copy()

if 'operation_date' in X_full.columns:
    op_date_full = pd.to_datetime(X_full['operation_date'], errors='coerce')
    X_full['operation_month'] = op_date_full.dt.month
    X_full['operation_day'] = op_date_full.dt.day

for col in ['scheduled_dispatch_time', 'scheduled_arrival_time', 'actual_dispatch_time']:
    if col in X_full.columns:
        decompose_hhmm(X_full, col)

X_full = X_full.drop(columns=['operation_date'], errors='ignore')

selected_pipeline.fit(X_full, y_full)
test_pred = selected_pipeline.predict_proba(test_model)[:, 1]

submission = pd.DataFrame({
    'transfer_id': test_df['transfer_id'],
    'predict_prob': test_pred,
})

submission.to_csv('submission.csv', index=False)
display(submission.head())
print('Saved submission.csv with required columns.')